# Animations — 60 m sprint

Every MP4 in this project, written to **`mp4s/`**. The analysis is in
`01_Angle_fPCA.ipynb` and the stage-by-stage checks are in
`sprint_pipeline.ipynb`; this notebook only renders.

The renderers live in **`sprint_outputs.py`** — sections 7 and 8 of the pipeline,
kept apart from the analysis so that importing the analysis can never touch a
plotting backend:

| function | what it draws | cleaning it uses |
|---|---|---|
| `render_trial` | one athlete, three panels | the **original** pipeline |
| `render_representative` | fast vs slow, one stride at top speed | the **analysis** pipeline |
| `render_representative_accel` | fast vs slow, blocks → 10 steps (~16 m) | the **analysis** pipeline |
| `render_representative_full` | fast vs slow, **the whole 62.5 m** | the **analysis** pipeline |

Output goes into folders by kind. Anything produced once **per athlete** gets its
own Title Case folder, so a set of thirty stays together:

```
mp4s/                the three group animations
mp4s/Trial Runs/     one video per athlete, all 30
```

`render_trial` is the original renderer ported across apart from the units, which
are now metres. Because it keeps its own cleaning its start frame differs from
`clean_for_angles`, so it will not line up frame-for-frame with the figures in
`01`. The three comparison videos use the analysis cleaning.

**These render smoothly now.** Every captured frame is drawn and played back at
the capture rate — 60 fps, real time — instead of every third frame at 20 fps;
the axes are pinned so a changing title cannot nudge them between frames; the
side-on camera follows a smoothed track rather than the stride-by-stride sway of
one marker; and the group loops drop their duplicate last sample so the join is
seamless. `sprint_outputs.py` has the detail at the top.

**One call does the lot.** `OUT.run_all_outputs()` renders every figure and every
athlete's MP4 — that is the intended way to produce the output stage, and this
notebook is for looking at one piece at a time.

Needs `ffmpeg` on the PATH.

In [1]:
from project_paths import PATHS

import numpy as np
import sprint_outputs as OUT
from sprint_pipeline import C3D_DIR, EXCLUDED_PIDS

print(f"writing to {OUT.MP4_OUT}")
print(f"{len(OUT.cohort_pids())} trials available, excluding {EXCLUDED_PIDS}")

writing to /Users/shunchen/Desktop/60m Project Folder/2026/08 Version/mp4s
30 trials available, excluding ['SB17']


## A · One trial

Velocity curve, the athlete side-on with the camera following, and the track from
above with the trail coloured by speed. The top-down panel doubles as an alignment
check: a correctly aligned trial runs in a straight horizontal line.

In [2]:
out = OUT.render_trial("SB153")
print(f"saved {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")

saved SB153_run.mp4  (2.1 MB)


## B · Representative fast against representative slow — top speed

The moving version of `pc1_anatomy.png`. Rather than one fast athlete and one slow
one, either of whom might be unusual, this averages the **3 fastest** into a single
skeleton and the **3 slowest** into another, then draws them over each other
through the stride cycle.

Each athlete contributes their 5 strides around peak velocity, time-normalised to
101 points. Scaled to 1.75 m **for drawing only**, so averaging athletes of
different sizes does not blur the mean — limb lengths survive it within about 1 %.

**Blue is the fast group, red the slow group.**

In [3]:
out = OUT.render_representative(n=3)
print(f"saved {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")

saved representative_top3_vs_bottom3.mp4  (1.9 MB)


## C · The same six, from the blocks through 10 steps

Acceleration is not top speed, and two things have to change.

**Steps cannot be averaged onto one cycle.** Every top-speed stride repeats the
last, so they collapse onto a single cycle cleanly. The first ten steps are a
*progression*: step 1 is nothing like step 10. So the **steps are the anchors** —
each is resampled to the same number of points, which guarantees "step 5" is
genuinely step 5 for every athlete however long they took to get there.

**Shape and position are separated.** The skeleton is scaled to 1.75 m so
different-sized athletes average into a clean body, but the pelvis sits at its
**true, unscaled** position — so the distance on the right panel is real.

| panel | shows |
|---|---|
| left | pelvis aligned — every difference is posture |
| right | true distance from the blocks — step length, and the gap opening |

**Read the clock, not just the gap.** Alignment is by step, so the two groups are
*not* at the same instant — the slower group takes longer to reach any given step.
Elapsed time is printed for each. It matters: SB202 is the slowest athlete at top
speed yet covers 16.9 m in ten steps, further than SB82's 14.6 m, simply by taking
3.55 s against 2.47 s.

In [4]:
out = OUT.render_representative_accel(n=3, n_steps=10)
print(f"saved {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")

saved representative_accel_top3_vs_bottom3.mp4  (2.9 MB)


## D · The whole run — where the gap opens

The acceleration video above stops at step 10, about 16 m in. This one runs the
trial out to the last step every athlete has — around **29 steps and 60 m** — and
holds the **entire track in one fixed view**, no camera following anybody. Both
groups are on it at once, so the distance between them is the result, printed as
a running gap.

The left panel is unchanged: pelvis-aligned posture, the same view as the
acceleration video.

One compromise, stated on the panel itself. 62 m of track against a 2 m athlete
is a 31:1 aspect, so drawn true to scale the runners are about forty pixels tall.
Each body is therefore drawn **4× life size about its own ground contact point**,
which leaves every position — and so every distance and the gap — exactly where
it really is. The right panel's vertical axis is unlabelled because of it.
`body_scale=1.0` gives the undistorted view.

In [5]:
out = OUT.render_representative_full(n=3)
print(f"saved {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")

saved representative_full_top3_vs_bottom3.mp4  (3.3 MB)


## E · Everyone

The output stage is meant to run the whole cohort — one video per athlete, plus
the three group animations. `run_all_outputs()` does the figures as well;
`trials=False` there skips the per-athlete videos.

`render_trial` still takes a single participant ID. `every=2` renders every
second frame for a quick look, and drops the frame rate with it so the video
stays real time.

In [6]:
# Everything, for everyone — about 20 s per athlete.
# written = OUT.run_all_outputs()

# out = OUT.render_trial("SB101", every=2)

for f in sorted(OUT.MP4_OUT.rglob("*.mp4")):
    rel = f.relative_to(OUT.MP4_OUT)
    print(f"  {str(rel):<48}{f.stat().st_size / 1e6:>6.1f} MB")

  Trial Runs/SB061_run.mp4                           1.5 MB
  Trial Runs/SB101_run.mp4                           1.5 MB
  Trial Runs/SB102_run.mp4                           1.8 MB
  Trial Runs/SB110_run.mp4                           1.6 MB
  Trial Runs/SB111_run.mp4                           1.5 MB
  Trial Runs/SB112_run.mp4                           1.5 MB
  Trial Runs/SB150_run.mp4                           1.8 MB
  Trial Runs/SB151_run.mp4                           1.8 MB
  Trial Runs/SB153_run.mp4                           2.1 MB
  Trial Runs/SB154_run.mp4                           1.6 MB
  Trial Runs/SB155_run.mp4                           1.5 MB
  Trial Runs/SB15_run.mp4                            1.7 MB
  Trial Runs/SB160_run.mp4                           1.9 MB
  Trial Runs/SB161_run.mp4                           1.7 MB
  Trial Runs/SB16_run.mp4                            1.7 MB
  Trial Runs/SB202_run.mp4                           1.9 MB
  Trial Runs/SB20_run.mp4               